# Pin Bar at Support/Resistance on SPY
## Strategy Brief
This strategy identifies pin bars at key support and resistance levels on the SPY ETF. A pin bar is a candlestick pattern that signals a potential reversal in price direction. The strategy predicts that a pin bar at support suggests a bullish reversal, while at resistance, it suggests a bearish reversal. Trades are executed based on these signals, with the expectation of capturing short-term price movements. The results are evaluated against a buy-and-hold strategy for comparison.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
In this phase, we define the parameters for our strategy, including the lookback period for support/resistance levels and the threshold for identifying pin bars.

In [ ]:
LOOKBACK_PERIOD = 20
PIN_BAR_THRESHOLD = 0.66

### PHASE 2 - Data Exploration
We will download historical SPY data from Yahoo Finance, calculate the necessary indicators to identify pin bars, and visualize them on the price chart.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start='2010-01-01')

# Calculate pin bar indicator
data['Range'] = data['High'] - data['Low']
data['Upper_Shadow'] = data['High'] - np.maximum(data['Open'], data['Close'])
data['Lower_Shadow'] = np.minimum(data['Open'], data['Close']) - data['Low']
data['Pin_Bar'] = ((data['Upper_Shadow'] > data['Range'] * PIN_BAR_THRESHOLD) | 
                  (data['Lower_Shadow'] > data['Range'] * PIN_BAR_THRESHOLD))

# Plot price and pin bars
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close')
plt.scatter(data.index[data['Pin_Bar']], data['Close'][data['Pin_Bar']], color='red', label='Pin Bar', marker='o')
plt.title('SPY Price with Pin Bars')
plt.legend()
plt.show()

### PHASE 3 - Strategy Engineering
In this phase, we define the trading signals based on pin bars at support and resistance levels. We will create a signal series and define the entry and exit logic.

In [ ]:
# Define support and resistance levels
data['Support'] = data['Low'].rolling(window=LOOKBACK_PERIOD).min()
data['Resistance'] = data['High'].rolling(window=LOOKBACK_PERIOD).max()

# Generate signals
signals = pd.Series(index=data.index, data=0)

# Long signal: pin bar at support
long_signal = (data['Pin_Bar'] & (data['Close'] < data['Support']))

# Short signal: pin bar at resistance
short_signal = (data['Pin_Bar'] & (data['Close'] > data['Resistance']))

signals[long_signal] = 1
signals[short_signal] = -1

# Positions based on signals
positions = signals.shift(1).fillna(0)

### PHASE 4 - Coding & Backtesting
We will backtest the strategy by calculating daily returns based on the positions and plot the equity curve.

In [ ]:
# Calculate daily returns
data['Returns'] = data['Close'].pct_change()

# Strategy returns
strategy_returns = positions * data['Returns']

# Equity curve
equity_curve = (1 + strategy_returns).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(equity_curve, label='Strategy Equity Curve')
plt.plot((1 + data['Returns']).cumprod(), label='Buy and Hold Equity Curve')
plt.title('Equity Curve')
plt.legend()
plt.show()

### PHASE 5 - Performance Evaluation
We will evaluate the strategy's performance using metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown, and compare it to a buy-and-hold strategy.

In [ ]:
def calculate_performance(equity_curve):
    # Calculate CAGR
    total_return = equity_curve.iloc[-1] / equity_curve.iloc[0] - 1
    years = (equity_curve.index[-1] - equity_curve.index[0]).days / 365.25
    cagr = (1 + total_return) ** (1 / years) - 1

    # Calculate Sharpe ratio
    sharpe_ratio = (strategy_returns.mean() / strategy_returns.std()) * np.sqrt(252)

    # Calculate Sortino ratio
    downside_returns = strategy_returns[strategy_returns < 0]
    sortino_ratio = (strategy_returns.mean() / downside_returns.std()) * np.sqrt(252)

    # Calculate Calmar ratio
    max_drawdown = (equity_curve / equity_curve.cummax() - 1).min()
    calmar_ratio = cagr / abs(max_drawdown)

    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

strategy_cagr, strategy_sharpe, strategy_sortino, strategy_calmar, strategy_max_dd = calculate_performance(equity_curve)
buy_hold_cagr, buy_hold_sharpe, buy_hold_sortino, buy_hold_calmar, buy_hold_max_dd = calculate_performance((1 + data['Returns']).cumprod())

# Compare with buy-and-hold
performance_comparison = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': [strategy_cagr, strategy_sharpe, strategy_sortino, strategy_calmar, strategy_max_dd],
    'Buy and Hold': [buy_hold_cagr, buy_hold_sharpe, buy_hold_sortino, buy_hold_calmar, buy_hold_max_dd]
})

performance_comparison

### PHASE 6 - Deploy & Monitor
We will create a function to download the last 60 days of SPY data, compute today's signal, and print the current position.

In [ ]:
def get_current_signal():
    recent_data = yf.download('SPY', period='60d')
    recent_data['Range'] = recent_data['High'] - recent_data['Low']
    recent_data['Upper_Shadow'] = recent_data['High'] - np.maximum(recent_data['Open'], recent_data['Close'])
    recent_data['Lower_Shadow'] = np.minimum(recent_data['Open'], recent_data['Close']) - recent_data['Low']
    recent_data['Pin_Bar'] = ((recent_data['Upper_Shadow'] > recent_data['Range'] * PIN_BAR_THRESHOLD) | 
                             (recent_data['Lower_Shadow'] > recent_data['Range'] * PIN_BAR_THRESHOLD))
    recent_data['Support'] = recent_data['Low'].rolling(window=LOOKBACK_PERIOD).min()
    recent_data['Resistance'] = recent_data['High'].rolling(window=LOOKBACK_PERIOD).max()

    # Determine today's signal
    today = recent_data.iloc[-1]
    if today['Pin_Bar']:
        if today['Close'] < today['Support']:
            print('Long Position')
        elif today['Close'] > today['Resistance']:
            print('Short Position')
        else:
            print('No Position')
    else:
        print('No Position')

get_current_signal()